In [1]:
import random as rand
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from BKT_pytorch import TorchBKT
import os
import joblib
import json
from ELO import Elo
import numpy as np

In [41]:
with open('../elo_variable.json', 'r') as Elo_Data:
    elo_data = json.load(Elo_Data)

print(elo_data)

{'globals': {'students': 500, 'init_skill_level': [0.0, 0.0], 'k_success': 1, 'k_fail': 0.5}, 'scenarios': [{'id': 1, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 0], [0, 1]], 'difficulty_level': [1, 1], 'depends': [-1, -1]}, {'id': 2, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 1], [1, 1]], 'difficulty_level': [0, 1], 'depends': [-1, -1]}, {'id': 3, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0], [0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1], 'depends': [-1, -1, -1, -1]}, {'id': 4, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1], [0, 1, 1, 1]], 'difficulty_level': [1, 1, 0, 1], 'depends': [-1, -1, -1, -1]}, {'id': 5, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1, 0, 1], [0, 1, 1, 0, 1, 1]], 'difficulty_level': [0, 0, 0, 1, 1, 1], 'depends': [-1, -1, -1, -1, -1, -1]}, {'id': 6, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1, 2, 2], 'depends': [-1, -1, -1, -1, -1, -1]}

In [42]:
def load_training_data(scenario, split):
    base_path = "../Simulated_Data"
    
    filename = f"scenario_{scenario}_{split}_data.csv"
    filepath = os.path.join(base_path, filename)
    
    return pd.read_csv(filepath)

In [43]:
all_data = {}

level_skill   = [] 
mastery_level = 1.5
i = 0

global_values = elo_data["globals"]
scenarios = elo_data["scenarios"]

students          = global_values["students"]
init_skill_level  = np.array(global_values["init_skill_level"])
k_success         = global_values["k_success"]

In [46]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    num_skills = scenario["num_skills"]
    train_df = load_training_data(scenario_id, "train")
    test_df = load_training_data(scenario_id, "test")

    print(f"Scenario {scenario_id}")

    model = TorchBKT()
    model.fit(train_df) 
    # Perform evaluation
    training_auc, train_acc = model.score(df=train_df)
    print(f"Scenario {scenario_id} train AUC, {training_auc} accuracy {train_acc}")
    model.save("../Trained_Models", scenario_id)
    # print(model.params())
    test_auc, test_acc = model.score(df=test_df)
    print(f"Scenario {scenario_id} test AUC, {test_auc} accuracy {test_acc}")

    # save_bkt(model, scenario_id, num_skills)

Scenario 1
fitting skill '0' (1 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.5537796117966108],
 'learn': 0.5327237248420715,
 'prior': 0.05479602888226509,
 'slip': [0.42209022501109816]}
fitting skill '1' (1 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.3443712702975725],
 'learn': 0.5564818382263184,
 'prior': 0.0487859807908535,
 'slip': [0.4821878677922036]}
Scenario 1 train AUC, 0.5971066991643285 accuracy 0.5727170236753101
[Scenario 1] TorchBKT saved → ../Trained_Models/TorchBKT_scenario_1.pth
Scenario 1 test AUC, 0.6480920060331825 accuracy 0.6
Scenario 2
fitting skill '0' (2 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.36342869009814566, 0.22697011165558886],
 'learn': 0.43101704120635986,
 'prior': 0.03159862384200096,
 'slip': [0.3878949156287305, 0.607571302407731]}
fitting skill '1' (2 tasks, 400 students)
{'forget': 0.0,
 'guess': [0.24993925077604734, 0.18972875987360785],
 'learn': 0.40954944491386414,
 'prior': 0.055043768137693405,
 'slip': [0.411685647